# AI Physics Tutor — Colab Runner (clean version)

MSc Thesis: **AI Assistant — A Multimodal Conversational Agent with Realistic Avatar for Physics Education**
Author: Kopalidis Thomas — MSc AI & DL, University of West Attica

## How to use
1. **Runtime → Change runtime type → T4 GPU**
2. Always run **Section 1 (Setup)** top to bottom in a new runtime.
3. Then run **only the section you need** (2 Smoke tests, 3 Benchmarks, 4 Gemini, 5 Avatar, 6 Live demo).
4. **Section 7** is troubleshooting — run only if something breaks.
5. Finish with **Section 8 (Stop)** so the runtime does not keep running.

Big files (models, PDFs, ChromaDB, results) live in Drive: `MyDrive/thesis_ai_and_dl`. Code comes from GitHub.

# 1. Setup (run every new runtime, top to bottom)

## Cell 1 — Verify runtime and GPU

In [58]:
import subprocess, sys, platform
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print('')
print('=== GPU ===')
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True)
    print(out.strip())
except Exception as e:
    print(f'nvidia-smi failed: {e}  (are you on a GPU runtime?)')
print('')
print('=== Disk ===')
print(subprocess.check_output(['df', '-h', '/content'], text=True).strip())

Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39

=== GPU ===
Tesla T4, 15360 MiB, 580.82.07

=== Disk ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   58G   55G  52% /


## Cell 2 — Mount Google Drive

Drive gives us persistent storage so Ollama model downloads and benchmark results survive runtime resets.

In [59]:
from google.colab import drive
drive.mount('/content/drive')

import os
THESIS_ROOT = '/content/drive/MyDrive/thesis_ai_and_dl'
os.makedirs(f'{THESIS_ROOT}/ollama_models', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/chroma_db', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/results', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/logs', exist_ok=True)
print(f'Thesis Drive folder: {THESIS_ROOT}')
!ls -la {THESIS_ROOT}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thesis Drive folder: /content/drive/MyDrive/thesis_ai_and_dl
total 4764
drwx------ 10 root root    4096 Oct 10 22:11 .
drwx------ 30 root root    4096 Oct 18 17:49 ..
drwx------  3 root root    4096 Oct 18 14:42 audio
drwx------  2 root root    4096 Oct 18 16:25 avatar
drwx------  3 root root    4096 Oct 18 19:40 chroma_db
-rw-------  1 root root 4836487 Oct 10 22:53 chunks.jsonl
drwx------  2 root root    4096 Oct 18 19:57 logs
drwx------  6 root root    4096 Oct 10 22:11 ollama_models
drwx------  2 root root    4096 Oct 10 22:43 openstax_pdfs
drwx------  2 root root    4096 Oct 10 22:11 results
drwx------  4 root root    4096 Oct 18 16:10 sadtalker


## Cell 3 — Clone the project repository

Clones into `/content/thesis_ai_and_dl` (fresh each runtime). Change `BRANCH` when needed.

In [73]:
import os, shutil
os.chdir('/content')                 # ← step out of the folder before deleting it
PROJECT_DIR = '/content/thesis_ai_and_dl'
REPO = 'https://github.com/thomaskopalidis/thesis_ai_and_dl.git'
BRANCH = 'avatar-interface'   # back to 'master' after merge

if os.path.exists(PROJECT_DIR):
    print(f'Removing existing {PROJECT_DIR}')
    shutil.rmtree(PROJECT_DIR)

!git clone --depth 1 -b {BRANCH} {REPO} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -5

Cloning into '/content/thesis_ai_and_dl'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 76 (delta 13), reused 58 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 4.10 MiB | 18.73 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/thesis_ai_and_dl
a36841f (grafted, HEAD -> avatar-interface, origin/avatar-interface) Created using Colab


## Cell 4 — Install Python dependencies

We pin versions known to work together (see docs/LAB_NOTEBOOK.md for the saga).
Colab usually has torch+cuda pre-installed; we skip reinstalling torch to avoid breaking the CUDA stack.

In [61]:
# Minimal install — only what Colab lacks. No version pins (use what resolves).
import subprocess, sys

packages_to_install = [
    "faster-whisper",
    "chromadb",
    "edge-tts",
    "pymupdf",
    "soxr",
    "einops",
]

print(f'Installing: {", ".join(packages_to_install)}')
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages_to_install,
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

print('\n=== Verification ===')
import importlib
for mod in ['torch', 'numpy', 'cv2', 'librosa', 'sentence_transformers',
            'chromadb', 'fastapi', 'faster_whisper', 'edge_tts',
            'soundfile', 'scipy', 'pkg_resources', 'einops']:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '?')
        print(f'OK   {mod:28s} {v}')
    except Exception as e:
        print(f'FAIL {mod}: {type(e).__name__}: {str(e)[:100]}')

Installing: faster-whisper, chromadb, edge-tts, pymupdf, soxr, einops

STDERR: Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/pip/__main__.py", line 8, in <module>
    if sys.path[0] in ("", os.getcwd()):
                           ~~~~~~~~~^^
FileNotFoundError: [Errno 2] No such file or directory


=== Verification ===
OK   torch                        2.11.0+cu130
OK   numpy                        2.1.3
OK   cv2                          4.14.0
OK   librosa                      0.11.0
OK   sentence_transformers        5.7.0
OK   chromadb                     1.5.9
OK   fastapi                      0.141.1
OK   faster_whisper               1.2.1
OK   edge_tts                     7.3.1
OK   soundfile                    0.14.0
OK   scipy                        1.16.3
OK   pkg_resources                ?
OK   einops                       0.8.2


## Cell 6 — Install ffmpeg (needed by STT and Wav2Lip output encoding)

In [62]:
!apt-get -qq update && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 >/dev/null
!ffmpeg -version | head -1

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## Cell 6b — Install Ollama (official installer)

Record the printed version in LAB_NOTEBOOK: all final benchmarks must use the same version.

In [63]:
import subprocess, time, os
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(2)
!apt-get install -y -qq zstd pciutils lshw >/dev/null
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
!ollama --version

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


## Cell 7 — Start Ollama server in background

Models are read from Drive (`OLLAMA_MODELS`), so they are downloaded only once.

In [64]:
import subprocess, time, requests, os
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(3)

env = os.environ.copy()
env['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '')
log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'
with open(log_path, 'ab') as f:
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'✓ Ollama ready after {i+1}s')
            print('Models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('[FAIL] Ollama did not start. Last log lines:')
    !tail -30 {log_path}

✓ Ollama ready after 4s
Models: ['phi3:mini', 'mistral:7b', 'llama3.2:3b', 'llama3.1:8b', 'qwen2.5:7b', 'qwen2.5:3b']


## Cell 8 — Pull the LLM models (only the first time; they stay in Drive)

In [65]:
MODELS_TO_PULL = [
    'qwen2.5:3b',      # 1.9 GB
    'qwen2.5:7b',      # 4.7 GB
    'llama3.1:8b',     # 4.9 GB
    'llama3.2:3b',     # 2.0 GB
    'mistral:7b',      # 4.1 GB
    'phi3:mini',       # 2.2 GB
    #'gemma3:1b',      # smoke tests only
    #'qwen2.5:14b-instruct-q4_K_M',  # only if GPU has >= 20 GB VRAM
]
for m in MODELS_TO_PULL:
    print(f'\n=== Pulling {m} ===')
    !ollama pull {m} 2>&1 | tail -2
print('\n=== Final list ===')
!ollama list


=== Pulling qwen2.5:3b ===
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
removing unused layers 
success 

=== Pulling qwen2.5:7b ===
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
removing unused layers 
success 

=== Pulling llama3.1:8b ===
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
removing unused layers 
success 

=== Pulling llama3.2:3b ===
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
removing unused layers 
success 

=== Pulling mistral:7b ===
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
removing unused layers 
success 

=== Pulling phi3:mini ===
shell-init: error retrieving current directory: getcwd: cannot access parent directori

## Cell 9 — Download OpenStax PDFs and build the KB index

The PDFs are not in git (too large). We download them once to Drive, run ingestion to produce `chunks.jsonl`, then build the ChromaDB index. All outputs live in Drive — next runtime restart will just reuse them.

In [66]:
import os, urllib.request

KB_DIR = f'{PROJECT_DIR}/Knowledge_base'
SRC_DIR = f'{KB_DIR}/sources'
DRIVE_PDFS = f'{THESIS_ROOT}/openstax_pdfs'
os.makedirs(DRIVE_PDFS, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

OPENSTAX = {
    'openstax_vol1.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume1-WEB_Zwi59QO.pdf',
    'openstax_vol2.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume2-WEB_oqeqFz9.pdf',
    'openstax_vol3.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume3-WEB_8rXeDQZ.pdf',
}

for fname, url in OPENSTAX.items():
    drive_path = f'{DRIVE_PDFS}/{fname}'
    project_path = f'{SRC_DIR}/{fname}'
    if not os.path.isfile(drive_path):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, drive_path)
    # Symlink from project dir into Drive copy — no duplication
    if not os.path.islink(project_path) and not os.path.isfile(project_path):
        os.symlink(drive_path, project_path)
    print(f'OK  {fname} ({os.path.getsize(drive_path)/1e6:.1f} MB)')

OK  openstax_vol1.pdf (81.5 MB)
OK  openstax_vol2.pdf (63.8 MB)
OK  openstax_vol3.pdf (53.5 MB)


In [67]:
# If a previously built chunks.jsonl exists in Drive, reuse it; otherwise re-ingest.
import os
DRIVE_CHUNKS = f'{THESIS_ROOT}/chunks.jsonl'
PROJECT_CHUNKS = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
os.makedirs(f'{PROJECT_DIR}/Knowledge_base/processed', exist_ok=True)

if os.path.isfile(DRIVE_CHUNKS):
    !cp {DRIVE_CHUNKS} {PROJECT_CHUNKS}
    print(f'Reused chunks.jsonl from Drive ({os.path.getsize(PROJECT_CHUNKS)/1e6:.1f} MB)')
else:
    !cd {PROJECT_DIR} && python evaluation/ingest_kb.py --chunk-size 220 --overlap 32 --preview 2
    !cp {PROJECT_CHUNKS} {DRIVE_CHUNKS}
    print('Fresh ingestion → saved to Drive for reuse')

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Reused chunks.jsonl from Drive (4.8 MB)


In [68]:
# Build (or reuse) ChromaDB index. Store the DB in Drive so it persists.
import os
DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# Symlink project's chroma_db -> Drive's chroma_db so build_index writes to Drive directly
if os.path.exists(PROJECT_CHROMA) and not os.path.islink(PROJECT_CHROMA):
    import shutil; shutil.rmtree(PROJECT_CHROMA)
if not os.path.islink(PROJECT_CHROMA):
    os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)

# Only build if the collection does not exist yet
import glob
if not glob.glob(f'{DRIVE_CHROMA}/*.sqlite3'):
    !cd {PROJECT_DIR} && python evaluation/build_index.py
else:
    print('ChromaDB index already present in Drive — skipping rebuild')
    !ls -la {DRIVE_CHROMA}

ChromaDB index already present in Drive — skipping rebuild
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
total 35856
drwx------  3 root root     4096 Oct 18 19:40 .
drwx------ 10 root root     4096 Oct 10 22:11 ..
drwx------  2 root root     4096 Oct 10 23:05 129b2ca1-0750-4ff3-8441-86e23fb87752
-rw-------  1 root root 36704256 Oct 18 19:20 chroma.sqlite3


## Cell 10 — Smoke test the RAG

In [74]:
import sys, os
for m in list(sys.modules):            # forget modules loaded from the deleted folder
    if m == 'config' or m == 'core' or m.startswith('core.'):
        del sys.modules[m]
print('app.py exists:', os.path.isfile(f'{PROJECT_DIR}/app.py'))
sys.path.insert(0, PROJECT_DIR)

r = retrieve_with_metadata("What is Newton's second law?", top_k=3)
for i, (cid, meta, dist, doc) in enumerate(zip(r['ids'], r['metadatas'], r['distances'], r['documents']), 1):
    print(f'[{i}] dist={dist:.3f}  {meta.get("chapter")}  /  {meta.get("section")}')
    print(f'    {doc[:140]}...')
    print()

app.py exists: True
[1] dist=0.385  Chapter 5 Newton's Laws of Motion  /  5.5 Newton’s Third Law
    Significance
To apply Newton’s second law as the primary equation in solving a problem, we sometimes have to rely on other
equations, such a...

[2] dist=0.410  Chapter 5 Newton's Laws of Motion  /  5.5 Newton’s Third Law
    When you push on a wall, the wall pushes back on you. This brings us to Newton’s third law. Newton’s third law represents a certain symmetry...

[3] dist=0.419  Chapter 5 Newton's Laws of Motion  /  5.3 Newton's Second Law
    If the magnitude of the truck’s acceleration is
 what is the magnitude of the sports
car’s acceleration? Component Form of Newton’s Second L...



##CELL 11 — Wav2Lip setup (για avatar)

In [75]:
import os
WAV2LIP_DIR = f'{PROJECT_DIR}/Wav2Lip'

if not os.path.exists(WAV2LIP_DIR):
    !cd {PROJECT_DIR} && git clone https://github.com/Rudrabha/Wav2Lip.git

# Patches
!sed -i 's|librosa.output.write_wav(path, wav, sr=sr)|import soundfile as _sf; _sf.write(path, wav, sr)|g' {WAV2LIP_DIR}/audio.py
!sed -i 's|librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,|librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,|g' {WAV2LIP_DIR}/audio.py

# Checkpoints
CKPT = f'{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth'
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 400_000_000:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    !wget -q -O {CKPT} 'https://huggingface.co/numz/wav2lip_studio/resolve/main/Wav2lip/wav2lip_gan.pth'

FACE_DET = f'{WAV2LIP_DIR}/face_detection/detection/sfd/s3fd.pth'
if not os.path.exists(FACE_DET):
    os.makedirs(os.path.dirname(FACE_DET), exist_ok=True)
    !wget -q -O {FACE_DET} 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth'

!pip install -q batch-face face-alignment
print(f'\n✓ Wav2Lip: {os.path.getsize(CKPT)/1e6:.0f} MB')
print(f'✓ FaceDet: {os.path.getsize(FACE_DET)/1e6:.0f} MB')


Cloning into 'Wav2Lip'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 409 (delta 2), reused 2 (delta 2), pack-reused 405 (from 2)
Receiving objects: 100% (409/409), 549.05 KiB | 17.71 MiB/s, done.
Resolving deltas: 100% (227/227), done.

✓ Wav2Lip: 436 MB
✓ FaceDet: 90 MB


## Cell 18.5 — Re-create chroma symlink + sync WER audio (D-014)

- **Chroma symlink**: idempotent, points the project's `chroma_db/` at the Drive copy.
- **WER audio**: calls `evaluation/sync_wer_audio.py` which enforces the "frozen reference set" policy from D-014.
  Behaviour:
  - If the Drive cache (`<THESIS_ROOT>/audio/synthetic/`) is complete and all SHA-256 hashes match the manifest → copy to workspace, skip generation.
  - If `edge-tts` version differs from the manifest → WARNING only (reference set stays frozen).
  - If anything is missing → ERROR. Re-run with `--regenerate` to archive the old set and build a new one.
  - First time ever → generate + mirror to Drive + write `audio_manifest.json`.

In [17]:
# Fix 1: Symlink chroma_db to Drive copy (idempotent)
import os, shutil, subprocess

DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

if os.path.exists(PROJECT_CHROMA) and not os.path.islink(PROJECT_CHROMA):
    shutil.rmtree(PROJECT_CHROMA)
if not os.path.islink(PROJECT_CHROMA):
    os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)
print(f'✓ Symlink: {PROJECT_CHROMA} → {DRIVE_CHROMA}')
!ls -lah {DRIVE_CHROMA}

# Fix 2: Sync WER audio from Drive cache (D-014)
DRIVE_AUDIO = f'{THESIS_ROOT}/audio/synthetic'
print(f'\n=== Syncing WER audio (Drive cache: {DRIVE_AUDIO}) ===')
os.chdir(PROJECT_DIR)
result = subprocess.run(
    ['python', 'evaluation/sync_wer_audio.py', '--drive', DRIVE_AUDIO],
    capture_output=True, text=True, timeout=300,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-800:])
    raise RuntimeError(
        f'sync_wer_audio.py exited with code {result.returncode}. '
        'If the Drive set is incomplete and you intend to rebuild it, '
        'rerun with --regenerate (archives the old set).'
    )
!ls {PROJECT_DIR}/evaluation/audio/synthetic/

✓ Symlink: /content/thesis_ai_and_dl/chroma_db → /content/drive/MyDrive/thesis_ai_and_dl/chroma_db
total 36M
drwx------  3 root root 4.0K Oct 18 17:52 .
drwx------ 10 root root 4.0K Oct 10 22:11 ..
drwx------  2 root root 4.0K Oct 10 23:05 129b2ca1-0750-4ff3-8441-86e23fb87752
-rw-------  1 root root  36M Oct 18 17:52 chroma.sqlite3

=== Syncing WER audio (Drive cache: /content/drive/MyDrive/thesis_ai_and_dl/audio/synthetic) ===
[OK] Copied 11 files from Drive, SHA-256 verified.

Q01.wav  Q05.wav  Q10.wav  Q16.wav  Q24.wav
Q03.wav  Q07.wav  Q13.wav  Q22.wav  Q28.wav


# 2. Smoke tests

## Cell 11 — Smoke test the LLM

One question through qwen2.5:3b to confirm Ollama + LLM wrapper work on GPU.

In [18]:
import time, os, requests, sys

MODEL = 'qwen2.5:3b'
os.environ['OLLAMA_MODEL'] = MODEL
os.environ['OLLAMA_READ_TIMEOUT'] = '300'  # 5 min for cold loads from Drive

# Clear cached python modules so new env vars take effect
for mod in list(sys.modules.keys()):
    if mod == 'config' or mod.startswith('core.'):
        del sys.modules[mod]
sys.path.insert(0, PROJECT_DIR)
import config
from core.llm import stream_llm_response

# --- Warm-up: ensure the model is loaded into VRAM before measuring ---
print(f'[WARMUP] Loading {MODEL} into VRAM (may take up to 60s on first call)...')
t_warm = time.time()
try:
    r = requests.post(
        'http://localhost:11434/api/generate',
        json={'model': MODEL, 'prompt': 'hi', 'stream': False,
              'options': {'num_predict': 1}, 'keep_alive': '30m'},
        timeout=(10, 300),
    )
    print(f'[WARMUP] model ready in {time.time()-t_warm:.1f}s (status {r.status_code})')
except Exception as e:
    print(f'[WARMUP] FAILED: {e}')
    raise

# --- Measured query ---
query = "What is Newton's second law?"
print(f'\n[QUERY] {query}')

t0 = time.time()
first_token_t = None
parts = []
for token in stream_llm_response(query, model=MODEL):
    if first_token_t is None:
        first_token_t = time.time() - t0
        print(f'[TIMING] first token: {first_token_t:.2f}s')
    parts.append(token)

answer = ''.join(parts).strip()
total = time.time() - t0
print(f'[TIMING] total: {total:.2f}s  ({len(parts)} tokens, {len(answer)} chars)')
print(f'\nA: {answer}')


[WARMUP] Loading qwen2.5:3b into VRAM (may take up to 60s on first call)...
[WARMUP] model ready in 130.8s (status 200)

[QUERY] What is Newton's second law?
[TIMING] first token: 6.16s
[TIMING] total: 19.92s  (28 tokens, 126 chars)

A: Newton's second law states that the force acting on an object is equal to the mass of the object times its acceleration. F=ma.


## Cell 12 — Smoke test TTS

Synthesise one short sentence to confirm edge-tts works in the sandbox.

In [19]:
import asyncio, time
from core.tts import text_to_speech_stream

async def _test():
    t0 = time.time()
    nbytes = 0
    async for chunk in text_to_speech_stream('Hello from Google Colab.'):
        nbytes += len(chunk)
    print(f'[TIMING] TTS: {time.time()-t0:.2f}s  ({nbytes} bytes of mp3 audio)')

await _test()

[TIMING] TTS: 0.70s  (14112 bytes of mp3 audio)


## Cell 13 — Status summary

Everything above should have run without errors. If so, we are ready for the benchmark script (next step).

In [20]:
import subprocess
print('=== Status ===')
print(f'Project:   {PROJECT_DIR}')
print(f'Drive:     {THESIS_ROOT}')
!which ollama
print('')
print('=== Loaded models (VRAM) ===')
!ollama ps
print('')
print('=== GPU utilisation ===')
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv,noheader

=== Status ===
Project:   /content/thesis_ai_and_dl
Drive:     /content/drive/MyDrive/thesis_ai_and_dl
/usr/local/bin/ollama

=== Loaded models (VRAM) ===
NAME          ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL               
qwen2.5:3b    357c53fb659c    2.1 GB    100% GPU     2048       llamacpp    29 minutes from now    

=== GPU utilisation ===
Tesla T4, 0 %, 2119 MiB, 15360 MiB


# 3. Benchmarks

## 3a — LLM + RAG only (`evaluation/benchmark.py`)
First a quick check (2 questions), then the full run.

In [21]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark.py \
    --models qwen2.5:3b \
    --rag both \
    --runs 1 \
    --warmups 1 \
    --limit-questions 2


/content/thesis_ai_and_dl/evaluation/benchmark.py:317: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
/content/thesis_ai_and_dl/evaluation/benchmark.py:217: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat(timespec="seconds") + "Z"
[2026-10-18T17:55:47Z] === Benchmark starting ===
[2026-10-18T17:55:47Z] Models:    ['qwen2.5:3b']
[2026-10-18T17:55:47Z] RAG:       both
[2026-10-18T17:55:47Z] Runs:      1 (+ 1 warm-up per config)
[2026-10-18T17:55:47Z] Max tok:   50
[2026-10-18T17:55:47Z] Top-K:     3
[2026-10-18T17:55:47Z] CSV out:   /content/thesis_ai_and_dl/evaluation/results/benchmark_20261018

In [22]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark.py \
    --models qwen2.5:3b qwen2.5:7b llama3.1:8b \
    --rag both \
    --runs 3 \
    --warmups 1


/content/thesis_ai_and_dl/evaluation/benchmark.py:317: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
/content/thesis_ai_and_dl/evaluation/benchmark.py:217: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat(timespec="seconds") + "Z"
[2026-10-18T17:56:08Z] === Benchmark starting ===
[2026-10-18T17:56:08Z] Models:    ['qwen2.5:3b', 'qwen2.5:7b', 'llama3.1:8b']
[2026-10-18T17:56:08Z] RAG:       both
[2026-10-18T17:56:08Z] Runs:      3 (+ 1 warm-up per config)
[2026-10-18T17:56:08Z] Max tok:   50
[2026-10-18T17:56:08Z] Top-K:     3
[2026-10-18T17:56:08Z] CSV out:   /content/thesis_ai_and_dl/evaluati

## 3b — Full pipeline (`evaluation/benchmark_full_pipeline.py`)
Smoke test (1 model, 2 questions), then all 6 models.

In [23]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark_full_pipeline.py \
    --models qwen2.5:3b \
    --runs 1 --warmups 1 \
    --limit-questions 2


[2026-10-18T18:21:24] === Full pipeline benchmark ===
[2026-10-18T18:21:24] Models:    ['qwen2.5:3b']
[2026-10-18T18:21:24] RAG:       on
[2026-10-18T18:21:24] Runs/Q:    1 (+1 warm-up per config)
[2026-10-18T18:21:24] Max tok:   50
[2026-10-18T18:21:24] CSV:       /content/thesis_ai_and_dl/evaluation/results/pipeline_20261018_182124.csv
[2026-10-18T18:21:24] 
[2026-10-18T18:21:24] ── Measuring STT (once, independent of LLM) ──
[2026-10-18T18:21:24] [STT] Warm-up on Q01.wav...
   [STT] Loading Whisper (base) on cuda...
   [STT] ✓ Whisper Model Ready!
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
[2026-10-18T18:21:29] [STT] Q01.wav: 0.378s
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
[2026-10-18T18:21:29] [STT] Q03.wav: 0.114s
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
[2026-10-18T18:21:30] [STT] Q05.wav: 0.107s
   ✗ ST

## Cell 20 Full Benchmark (6 models)
1. qwen2.5:3b
2. qwen2.5:7b
3. llama3.1:8b
4. mistral:7b
5. phi3:mini
6. llama3.2:3b

In [24]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark_full_pipeline.py \
    --models qwen2.5:3b qwen2.5:7b llama3.1:8b mistral:7b phi3:mini llama3.2:3b \
    --runs 1 --warmups 1

[2026-10-18T18:22:04] === Full pipeline benchmark ===
[2026-10-18T18:22:04] Models:    ['qwen2.5:3b', 'qwen2.5:7b', 'llama3.1:8b', 'mistral:7b', 'phi3:mini', 'llama3.2:3b']
[2026-10-18T18:22:04] RAG:       on
[2026-10-18T18:22:04] Runs/Q:    1 (+1 warm-up per config)
[2026-10-18T18:22:04] Max tok:   50
[2026-10-18T18:22:04] CSV:       /content/thesis_ai_and_dl/evaluation/results/pipeline_20261018_182204.csv
[2026-10-18T18:22:04] 
[2026-10-18T18:22:04] ── Measuring STT (once, independent of LLM) ──
[2026-10-18T18:22:04] [STT] Warm-up on Q01.wav...
   [STT] Loading Whisper (base) on cuda...
   [STT] ✓ Whisper Model Ready!
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
[2026-10-18T18:22:06] [STT] Q01.wav: 0.112s
   ✗ STT Error: open() got an unexpected keyword argument 'metadata_errors'
[2026-10-18T18:22:06] [STT] Q03.wav: 0.114s
   ✗ STT Error: open() got an unexpected keyword argumen

## 3c — Results table (300 dpi)
Change `summary_file` to the CSV you want to plot.

In [25]:
import matplotlib.pyplot as plt

summary_file = f'{PROJECT_DIR}/evaluation/results/stages_summary_20261011_104315.csv'
df = pd.read_csv(summary_file)

# Prepare display
df = df[['model', 'stt_s', 'rag_s', 'llm_s', 'tts_s', 'avatar_s', 'total_s']]
df = df.round({'stt_s': 3, 'rag_s': 3, 'llm_s': 2, 'tts_s': 2, 'avatar_s': 2, 'total_s': 2})
df = df.sort_values('total_s').reset_index(drop=True)

# Rename columns for publication
df.columns = ['Model', 'STT (s)', 'RAG (s)', 'LLM (s)', 'TTS (s)', 'Avatar (s)', 'Total (s)']

# Render table as image
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
tbl = ax.table(cellText=df.values, colLabels=df.columns,
               loc='center', cellLoc='center',
               colColours=['#4a90e2']*len(df.columns))

# Styling
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.1, 2.0)

# Make header white text
for i in range(len(df.columns)):
    tbl[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(df)+1):
    for j in range(len(df.columns)):
        if i % 2 == 0:
            tbl[(i, j)].set_facecolor('#f0f0f0')

plt.title('Full Pipeline Benchmark — 6 LLMs on NVIDIA T4 (30 questions, RAG=on)',
          fontsize=13, fontweight='bold', pad=15)
plt.savefig('/content/stages_table.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('\n✓ Saved to /content/stages_table.png')

FileNotFoundError: [Errno 2] No such file or directory: '/content/thesis_ai_and_dl/evaluation/results/stages_summary_20261011_104315.csv'

# 4. Cloud LLM (Gemini)

The key lives in Colab **Secrets** (key icon on the left) as `GEMINI_API_KEY`. Never print the key.

In [26]:
from google.colab import userdata
key = userdata.get('GEMINI_API_KEY').strip()
print(f'Key loaded: length={len(key)}, valid prefix={key.startswith(("AIza", "AQ."))}')

Key loaded: length=53, valid prefix=True


## Cell 22 (quick Gemini API test)

In [27]:
from google.colab import userdata
import requests

key = userdata.get('GEMINI_API_KEY').strip()

# Try newer models
for model in ['gemini-3.8-flash', 'gemini-3.8-pro', 'gemini-3.5-flash']:
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={key}'
    try:
        r = requests.post(url, json={
            'contents': [{'parts': [{'text': 'Say hello in one word.'}]}]
        }, timeout=30)
        if r.ok:
            print(f'✓ {model}: {r.json()["candidates"][0]["content"]["parts"][0]["text"]}')
        else:
            err = r.json().get('error', {}).get('message', r.text[:200])
            print(f'✗ {model}: {r.status_code} - {err[:150]}')
    except Exception as e:
        print(f'✗ {model}: {e}')


✓ gemini-3.8-flash: Hello.
✗ gemini-3.8-pro: 404 - models/gemini-3.8-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of av
✓ gemini-3.5-flash: Hello


## Cell 24 Quick Gemini test μέσω του project wrapper

In [28]:
import os, sys
from google.colab import userdata

# Set env var so core/llm.py picks it up
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Clear module cache for fresh config
for mod in list(sys.modules.keys()):
    if mod == 'config' or mod.startswith('core.'):
        del sys.modules[mod]

sys.path.insert(0, PROJECT_DIR)
from core.llm import stream_llm_response
import time

t0 = time.time()
parts = []
for tok in stream_llm_response("What is Newton's second law?", model='cloud:google:gemini-3.5-flash'):
    parts.append(tok)
answer = ''.join(parts).strip()
print(f'[TIMING] {time.time()-t0:.2f}s')
print(f'A: {answer}')


[TIMING] 1.48s
A: Cloud error (google): unexpected response {'candidates': [{'content': {}, 'finishReason': 'MAX_TOKENS', 'index': 0}], 'usageMetadata': {'promptTokenCount': 72, 'totalTokenCount': 148, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount':


## Cell 25 - Benchmark Gemini

In [29]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

!python evaluation/benchmark_full_pipeline.py \
    --models cloud:google:gemini-3.5-flash \
    --runs 1 --warmups 1 \
    --skip-stt --skip-avatar


TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

# 5. Avatar experiments (Wav2Lip / SadTalker / MuseTalk)

Uses the common `AvatarEngine` interface (`core/avatar_engine.py`). Needs Section 1 (Wav2Lip setup + wavs). Ollama is not needed here (D-011: stop it during avatar measurements).

## 5.0 — Test 3b: Wav2Lip on a VIDEO base

In [30]:
import os, sys
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
for m in list(sys.modules):                      # fresh imports
    if m == 'config' or m.startswith('core'):
        del sys.modules[m]
from core import avatar_engine
print('Available engines:', avatar_engine.available_engines())

AUDIO = 'evaluation/audio/synthetic/Q05.wav'
assert os.path.isfile(AUDIO), 'Run the "regenerate wavs" cell in Section 1 first'

# 3-second test video, 25 fps, aspect ratio preserved
!ffmpeg -y -loglevel error -loop 1 -i frontend/assets/teacher_avatar.jpg -t 3 -r 25 -vf "scale=-2:512" -pix_fmt yuv420p -c:v libx264 _test_video.mp4

r = avatar_engine.get('wav2lip').generate(AUDIO, base_path='_test_video.mp4')
print(f'base_kind={r.base_kind} time={r.time_s:.2f}s dur={r.video_duration_s:.2f}s '
      f'fps={r.fps:.1f} vram={r.peak_vram_mb:.0f}MB encode={r.encode_time_s:.2f}s')

from IPython.display import Video
Video(r.video_path, embed=True, width=320)

Available engines: ['wav2lip']
[Wav2Lip] Prepared video: /content/thesis_ai_and_dl/_test_video.mp4
base_kind=video time=58.95s dur=2.76s fps=25.0 vram=10096MB encode=0.20s


## 5.1 — Idle video with SadTalker (STEP 1)

One-off generation of a silent idle video of the teacher avatar (D-015). The MuseTalk pipeline (STEP 2) will lip-sync onto this video.

**Design decisions**
- **Source image**: `teacher_avatar_v2.jpg` (1792×2400, AI-generated, SHA-256 recorded in D-016). Aspect-preserving resize to max side 1200 px inside the SadTalker adapter.
- **Driving audio**: silent 4–10 s @ 16 kHz mono. Low pink noise (`anoisesrc=c=pink:a=0.003`) or pure silence (`anullsrc`) — compared visually in stage 2.
- **Post-processing**: ping-pong loop (forward + reverse). Reversal point is the same frame, so the seam is seamless.
- **Frozen asset policy**: candidate videos live under `avatar/candidates/<name>.mp4` + `<name>.json`. The chosen variant is `--promote`d to `avatar/teacher_idle_25fps.mp4` + `teacher_idle_manifest.json`. Any previous frozen asset is archived (not deleted).

**Pipeline**
1. **Setup cell** — clones SadTalker, downloads checkpoints + GFPGAN weights to Drive, applies known patches for new torch/numpy. One-off; skipped if already present.
2. **Stage 1 (preprocess)** — 3 candidates × 4 s to pick `full` vs `extcrop` vs `full+still`.
3. **Stage 2 (audio)** — in the chosen preprocess, pick `pink` vs `silence`.
4. **Final (enhancer)** — chosen combo at 10 s, with and without GFPGAN.
5. **Promote** — mark the chosen candidate as the frozen asset.

In [31]:
 — SadTalker setup (clone + checkpoints + GFPGAN + patches)
# One-off per Colab runtime. Checkpoints go to Drive to survive runtime restarts.
import os, subprocess, time, urllib.request
from pathlib import Path

SADTALKER_DIR = Path(f"{PROJECT_DIR}/SadTalker")
DRIVE_CKPTS   = Path(f"{THESIS_ROOT}/sadtalker/checkpoints")
DRIVE_GFPGAN  = Path(f"{THESIS_ROOT}/sadtalker/gfpgan_weights")

# 1. Clone SadTalker if missing
if not SADTALKER_DIR.exists():
    print("Cloning SadTalker…")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/OpenTalker/SadTalker.git",
                    str(SADTALKER_DIR)], check=True)
else:
    print(f"✓ SadTalker already cloned: {SADTALKER_DIR}")

# 2. Download checkpoints to Drive (one-off), symlink into SadTalker/checkpoints
DRIVE_CKPTS.mkdir(parents=True, exist_ok=True)
CKPTS = {
    "mapping_00109-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar",
    "mapping_00229-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar",
    "SadTalker_V0.0.2_256.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors",
    "SadTalker_V0.0.2_512.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors",
}
for name, url in CKPTS.items():
    dst = DRIVE_CKPTS / name
    if not dst.is_file() or dst.stat().st_size < 1_000_000:
        print(f"  ↓ {name}")
        urllib.request.urlretrieve(url, dst)
PROJECT_CKPTS = SADTALKER_DIR / "checkpoints"
if PROJECT_CKPTS.is_symlink() or PROJECT_CKPTS.exists():
    if PROJECT_CKPTS.is_symlink():
        PROJECT_CKPTS.unlink()
    elif PROJECT_CKPTS.is_dir():
        import shutil as _sh; _sh.rmtree(PROJECT_CKPTS)
os.symlink(DRIVE_CKPTS, PROJECT_CKPTS)
print(f"✓ Checkpoints symlinked: {PROJECT_CKPTS} → {DRIVE_CKPTS}")

# 3. GFPGAN weights (face enhancer). Cached in Drive, symlinked to SadTalker/gfpgan/weights.
DRIVE_GFPGAN.mkdir(parents=True, exist_ok=True)
GFPGAN_WEIGHTS = {
    "alignment_WFLW_4HG.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
    "detection_Resnet50_Final.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
    "GFPGANv1.4.pth": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    "parsing_parsenet.pth": "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
}
for name, url in GFPGAN_WEIGHTS.items():
    dst = DRIVE_GFPGAN / name
    if not dst.is_file() or dst.stat().st_size < 100_000:
        print(f"  ↓ {name}")
        urllib.request.urlretrieve(url, dst)
PROJECT_GFPGAN = SADTALKER_DIR / "gfpgan" / "weights"
PROJECT_GFPGAN.parent.mkdir(parents=True, exist_ok=True)
if PROJECT_GFPGAN.is_symlink() or PROJECT_GFPGAN.exists():
    if PROJECT_GFPGAN.is_symlink():
        PROJECT_GFPGAN.unlink()
    elif PROJECT_GFPGAN.is_dir():
        import shutil as _sh; _sh.rmtree(PROJECT_GFPGAN)
os.symlink(DRIVE_GFPGAN, PROJECT_GFPGAN)
print(f"✓ GFPGAN weights symlinked: {PROJECT_GFPGAN} → {DRIVE_GFPGAN}")

# 4. Install SadTalker python deps (Python 3.13-safe)
import sys, tarfile, importlib.util

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-3000:])
        raise RuntimeError("pip failed: " + " ".join(args))

# 4a. Plain deps (wheels, no build issues)
pip("safetensors", "yacs", "pydub", "resampy", "kornia", "imageio-ffmpeg",
    "face-alignment", "addict", "lmdb", "yapf", "filterpy", "numba")

# 4b. basicsr 1.4.2: setup.py breaks on Python 3.13 (locals() change) -> patch + install
if importlib.util.find_spec("basicsr") is None:
    src = Path("/content/_basicsr"); src.mkdir(exist_ok=True)
    tgz = src / "basicsr-1.4.2.tar.gz"
    urllib.request.urlretrieve(
        "https://files.pythonhosted.org/packages/source/b/basicsr/basicsr-1.4.2.tar.gz", tgz)
    tarfile.open(tgz).extractall(src)
    setup_py = src / "basicsr-1.4.2" / "setup.py"
    s = setup_py.read_text()
    s = s.replace("exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']",
                  "_ns = {}; exec(compile(f.read(), version_file, 'exec'), _ns)\n    return _ns['__version__']")
    setup_py.write_text(s)
    pip("--no-deps", "--no-build-isolation", str(src / "basicsr-1.4.2"))
    print("✓ basicsr 1.4.2 installed (patched for Python 3.13)")

# 4c. gfpgan + facexlib without deps (they would try to rebuild basicsr)
pip("--no-deps", "gfpgan", "facexlib")
print("✓ pip deps installed")
# 5. Known patches for new torch/numpy compatibility
# 5a. basicsr: torchvision.transforms.functional_tensor → torchvision.transforms.functional
import glob
for pyfile in glob.glob("/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py"):
    subprocess.run(["sed", "-i",
                    "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/",
                    pyfile], check=False)
    print(f"✓ patched {pyfile}")

# 5b. numpy: np.float / np.int / np.complex removed in numpy>=1.20
import re
for pyfile_pattern in [
    "/usr/local/lib/python*/dist-packages/basicsr/metrics/niqe.py",
    f"{SADTALKER_DIR}/src/face3d/util/preprocess.py",
    f"{SADTALKER_DIR}/src/face3d/util/load_mats.py",
]:
    for pyfile in glob.glob(pyfile_pattern):
        txt = Path(pyfile).read_text()
        txt2 = re.sub(r"\bnp\.float\b(?!\d)", "np.float64", txt)
        txt2 = re.sub(r"\bnp\.int\b(?!\d)",   "np.int64",   txt2)
        txt2 = re.sub(r"\bnp\.complex\b(?!\d)", "np.complex128", txt2)
        if txt2 != txt:
            Path(pyfile).write_text(txt2)
            print(f"✓ patched np.* in {pyfile}")

# 5c. SadTalker preprocess.py: numpy>=1.24 "inhomogeneous shape" fix.
#     Replace "t[0], t[1]])" with "t[0][0], t[1][0]])" so arrays become scalars.
_preprocess = SADTALKER_DIR / "src" / "face3d" / "util" / "preprocess.py"
if _preprocess.is_file():
    _old = "t[0], t[1]])"
    _new = "t[0][0], t[1][0]])"
    _txt = _preprocess.read_text()
    if _old in _txt and _new not in _txt:
        _preprocess.write_text(_txt.replace(_old, _new))
        print(f"✓ patched inhomogeneous-shape in {_preprocess}")
    elif _new in _txt:
        print(f"✓ already patched (inhomogeneous-shape): {_preprocess}")

# 6. Verify adapter sees SadTalker as available
for m in list(__import__("sys").modules):
    if m == "config" or m.startswith("core"):
        del __import__("sys").modules[m]
from core.engines.sadtalker import SadTalkerEngine
print(f"\n✓ SadTalkerEngine.is_available(): {SadTalkerEngine().is_available()}")


Cloning SadTalker…
✓ Checkpoints symlinked: /content/thesis_ai_and_dl/SadTalker/checkpoints → /content/drive/MyDrive/thesis_ai_and_dl/sadtalker/checkpoints
✓ GFPGAN weights symlinked: /content/thesis_ai_and_dl/SadTalker/gfpgan/weights → /content/drive/MyDrive/thesis_ai_and_dl/sadtalker/gfpgan_weights


/tmp/ipykernel_1026/2873074332.py:84: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(tgz).extractall(src)


✓ basicsr 1.4.2 installed (patched for Python 3.13)
✓ pip deps installed
✓ patched /usr/local/lib/python3.13/dist-packages/basicsr/data/degradations.py
✓ patched inhomogeneous-shape in /content/thesis_ai_and_dl/SadTalker/src/face3d/util/preprocess.py

✓ SadTalkerEngine.is_available(): True


In [32]:
# 5.1a-fix — NumPy 2.x: patch removed aliases across all SadTalker .py files (idempotent)
import re
from pathlib import Path
SADTALKER_DIR = Path(f"{PROJECT_DIR}/SadTalker")
FIXES = [
    (r"\bnp\.VisibleDeprecationWarning\b", "np.exceptions.VisibleDeprecationWarning"),
    (r"\bnp\.float\b",   "np.float64"),
    (r"\bnp\.int\b",     "np.int64"),
    (r"\bnp\.complex\b", "np.complex128"),
    (r"\bnp\.bool\b",    "np.bool_"),
]
n = 0
for f in SADTALKER_DIR.rglob("*.py"):
    txt = f.read_text(encoding="utf-8", errors="ignore")
    new = txt
    for pat, rep in FIXES:
        new = re.sub(pat, rep, new)
    if new != txt:
        f.write_text(new, encoding="utf-8"); n += 1
print(f"✓ NumPy-2 patch: {n} file(s) changed")

✓ NumPy-2 patch: 6 file(s) changed


In [ ]:
# Cell 5.1b — Stage 1: 3 candidates to pick preprocess mode (4 s each)
# Expected runtime: ~2-4 min each on T4 (no enhancer).
import os, subprocess
os.chdir(PROJECT_DIR)

DRIVE_AVATAR = f"{THESIS_ROOT}/avatar"
os.makedirs(DRIVE_AVATAR, exist_ok=True)

CANDIDATES = [
    ("s1_full",       ["--preprocess", "full",    "--no-still"]),
    ("s1_extcrop",    ["--preprocess", "extcrop", "--no-still"]),
    ("s1_full_still", ["--preprocess", "full",    "--still"]),
]

for name, extra in CANDIDATES:
    print(f"\n{'='*60}\n>>> {name}\n{'='*60}")
    cand_json = f"{DRIVE_AVATAR}/candidates/{name}.json"
    if os.path.isfile(cand_json):
        print(f"[SKIP] {cand_json} already exists")
        continue
    cmd = ["python", "evaluation/make_idle_video.py",
           "--drive", DRIVE_AVATAR, "--name", name,
           "--audio", "pink", "--enhancer", "none", "--duration", "4",
           "--expression-scale", "0.5", "--pose-style", "0", "--size", "256",
          ] + extra
    # Stream output live (capture_output or default subprocess.run blocks until done in Colab).
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        print(f"[FAIL] {name} exited with code {rc}")
        break


In [ ]:
# Cell 5.1c — Side-by-side viewer for candidates of a given stage
# Videos are embedded as base64 data URIs (Colab cannot play /content/* in <video>).
# Change STAGE_PREFIX to inspect different stages ("s1_", "s2_", "final_").
import base64, json
from pathlib import Path
from IPython.display import HTML, display

DRIVE_AVATAR = Path(f"{THESIS_ROOT}/avatar")
STAGE_PREFIX = "s2_"   # ← change to "s2_" or "final_" as needed

candidates = sorted(DRIVE_AVATAR.glob(f"candidates/{STAGE_PREFIX}*.mp4"))
if not candidates:
    print(f"No candidates matching '{STAGE_PREFIX}*' under {DRIVE_AVATAR}/candidates/")
else:
    html = "<table style='border-spacing:12px'><tr>"
    for src in candidates:
        name = src.stem
        meta_file = src.with_suffix(".json")
        meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}
        cfg = meta.get("sadtalker_config", {})
        meas = meta.get("measurements", {})
        info = (f"preprocess={cfg.get('preprocess')} · still={cfg.get('still')} · "
                f"audio={cfg.get('audio_kind')} · enhancer={cfg.get('enhancer')}<br>"
                f"{meta.get('video_width')}×{meta.get('video_height')} · "
                f"{meta.get('video_duration_s', 0):.1f}s · "
                f"SadTalker {meas.get('sadtalker_time_s', '?')}s · "
                f"VRAM {meas.get('sadtalker_peak_vram_mb', '?')}MB")
        data = base64.b64encode(src.read_bytes()).decode()
        html += (f"<td style='vertical-align:top;text-align:center'>"
                 f"<b>{name}</b><br>"
                 f"<video src='data:video/mp4;base64,{data}' "
                 f"width='300' controls autoplay loop muted></video><br>"
                 f"<span style='font-size:11px;color:#666'>{info}</span>"
                 f"</td>")
    html += "</tr></table>"
    display(HTML(html))
    print(f"\nShowing {len(candidates)} candidate(s). Pick one, then run the stage-2 cell.")


## 5.2 — MuseTalk (STEP 2, to be added)

# 6. Live demo

##CELL 12 — LIVE DEMO

In [76]:
# CELL 12 — LIVE DEMO
import subprocess, os, time, requests
from google.colab import userdata, output
from IPython.display import HTML, display

# ─── Settings ───
os.environ['TTS_VOICE'] = 'en-GB-SoniaNeural'   # ← φωνή της δασκάλας

# ─── Kill previous server ───
subprocess.run(['pkill', '-9', '-f', 'app.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], check=False)
time.sleep(2)

# ─── Load cloud API keys from Secrets (never printed) ───
for name in ['GEMINI_API_KEY', 'OPENROUTER_API_KEY', 'GROQ_API_KEY',
             'OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    try:
        os.environ[name] = userdata.get(name)
        print(f'✓ {name}')
    except Exception:
        print(f'  (skip {name})')

# ─── Launch app.py ───
os.chdir(PROJECT_DIR)
print('\n🚀 Launching app.py...')
log_path = f'{THESIS_ROOT}/logs/app_server.log'
log = open(log_path, 'wb')
proc = subprocess.Popen(['python', 'app.py'], stdout=log, stderr=subprocess.STDOUT)

# ─── Wait for server ready ───
ready = False
for i in range(90):
    try:
        r = requests.get('http://localhost:8000/api/models', timeout=1)
        if r.ok:
            print(f'✓ Server ready after {i+1}s')
            print(f'  Models: {r.json()["models"]}')
            ready = True
            break
    except Exception:
        pass
    if proc.poll() is not None:          # server crashed
        break
    time.sleep(1)

if not ready:
    print('\n[FAIL] Server did not start. Last log lines:')
    log.flush()
    !tail -30 {log_path}
else:
    # ─── Check voice actually loaded by the server ───
    env = open(f'/proc/{proc.pid}/environ', 'rb').read().decode(errors='ignore').split('\0')
    print('  Voice:', [e for e in env if e.startswith('TTS_VOICE=')])

        # ─── Public HTTPS link via Cloudflare quick tunnel (no account needed) ───
    import re
    CF = '/content/cloudflared'
    if not os.path.exists(CF):
        !wget -q -O {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x {CF}
    subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
    cf_log = '/content/cloudflared.log'
    subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'],
                     stdout=open(cf_log, 'w'), stderr=subprocess.STDOUT)
    public = None
    for _ in range(30):
        time.sleep(1)
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(cf_log).read())
        if m:
            public = m.group(0) + '/app/'
            break
    if public:
        display(HTML(f'''
          <a href="{public}" target="_blank"
             style="display:inline-block;padding:14px 28px;background:#7c3aed;color:#fff;
                    border-radius:10px;font-size:20px;font-weight:600;text-decoration:none">
             🎬 Άνοιξε το AI Physics Tutor</a>
          <p style="color:#888">{public}<br>⚠️ Δημόσιο link: μην το μοιράζεσαι.</p>'''))
    else:
        print('[FAIL] Tunnel did not start:'); print(open(cf_log).read()[-1500:])
    print(f'📊 Logs: !tail -40 {log_path}')

✓ GEMINI_API_KEY
✓ OPENROUTER_API_KEY
✓ GROQ_API_KEY
  (skip OPENAI_API_KEY)
  (skip ANTHROPIC_API_KEY)

🚀 Launching app.py...
✓ Server ready after 10s
  Models: ['phi3:mini', 'mistral:7b', 'llama3.2:3b', 'llama3.1:8b', 'qwen2.5:7b', 'qwen2.5:3b']
  Voice: ['TTS_VOICE=en-GB-SoniaNeural']


📊 Logs: !tail -40 /content/drive/MyDrive/thesis_ai_and_dl/logs/app_server.log


## Cell bonus here different voice

In [37]:
import edge_tts
from IPython.display import Audio, display
TEXT = "Newton's second law states that the net force on an object equals its mass times its acceleration."
for v in ["en-US-JennyNeural", "en-US-AriaNeural", "en-GB-SoniaNeural",
          "en-GB-LibbyNeural", "en-IE-EmilyNeural"]:
    path = f"/content/voice_{v}.mp3"
    await edge_tts.Communicate(TEXT, v).save(path)
    print(v); display(Audio(path))

en-US-JennyNeural


en-US-AriaNeural


en-GB-SoniaNeural


en-GB-LibbyNeural


en-IE-EmilyNeural


### Alternative public URL via ngrok (needs `NGROK_TOKEN` in Secrets). Do not share the link.

In [39]:
!pip install -q pyngrok
from pyngrok import ngrok, conf
from google.colab import userdata
conf.get_default().auth_token = userdata.get('NGROK_TOKEN')
ngrok.kill()
public_url = ngrok.connect(8000, "http")
demo_url = f"{public_url.public_url}/app/"
print(f"\n{'='*60}")
print(f"🎬 LIVE DEMO URL:")
print(f"{demo_url}")
print(f"{'='*60}\n")


LLM model set to: qwen2.5:3b
Avatar mode set to: True
Lipsync engine set to: sadtalker
User (text): What is Newton’s second law?
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5828.41it/s]
[RAG] Loaded existing collection: 3950 chunks
Assistant is thinking & speaking... [engine: sadtalker]
[TIMING] Time to first LLM token: 11.83s
[TIMING] TTS(edge) sentence #1: 0.76s
[SadTalker] Prepared source: /tmp/tmpt3uzh_uc_sadtalker_src.jpg (896x1200)
[TIMING] sadtalker sentence #1: 236.69s
[TIMING] TTS(edge) sentence #2: 0.72s
[TIMING] sadtalker sentence #2: 161.36s
[TIMING] TTS(edge) sentence #3: 0.67s
[TIMING] sadtalker sentence #3: 223.49s
[TIMING] TTS(edge) sentence #4: 0.65s
[TIMING] sadtalker sentence #4: 97.06s
[TIMING] SUMMARY -> total: 733.64s | LLM: 733.64s | TTS total: 2.80s | sadtalker total: 718.60s | sentences: 4
INFO:     2a02:1388:a4a:4fbf:3484:dbba:46a2:197d:0 - "GET /app/assets/last_response.mp4?t=1792350048873 HTTP/1.1" 206 Partial Content
INFO:     2a02:1388:a4a:4fbf

KeyboardInterrupt: 

### Restart the app after a `git pull`

In [ ]:
%cd /content/thesis_ai_and_dl
!git pull
!pkill -f "python3 app.py"
import time, subprocess
time.sleep(3)
subprocess.Popen(['python3', 'app.py'],
                 stdout=open('/tmp/app.log','w'),
                 stderr=subprocess.STDOUT,
                 cwd='/content/thesis_ai_and_dl')
time.sleep(6)
print("✓ Server restarted")
!tail -10 /tmp/app.log


##CELL 13 — Stop server

In [ ]:
import subprocess
subprocess.run(['pkill', '-9', '-f', 'app.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], check=False)
print('✓ Server stopped')


# 7. Troubleshooting (run ONLY when something is broken)

### Is Ollama alive? Which models?

In [ ]:
import os, requests
print('=== Variables ===')
try:
    print(f'PROJECT_DIR: {PROJECT_DIR}')
    print(f'THESIS_ROOT: {THESIS_ROOT}')
except NameError as e:
    print(f'FAIL: {e}')

print('\n=== Ollama ===')
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    print(f'OK, models: {[m["name"] for m in r.json().get("models", [])]}')
except Exception as e:
    print(f'DEAD: {e}')


### Ollama logs + GPU state

In [ ]:
# Get Ollama server logs
!tail -60 {THESIS_ROOT}/logs/ollama_serve.log

print('\n\n=== GPU state ===')
!nvidia-smi

print('\n=== CUDA version ===')
!nvcc --version 2>/dev/null || echo "nvcc not installed"

print('\n=== Ollama version ===')
!ollama --version


### Models loaded in VRAM

In [ ]:
import subprocess, requests

# Check if any model is loaded
print('=== Models loaded in GPU/CPU ===')
try:
    r = requests.get('http://localhost:11434/api/ps', timeout=3)
    print(r.json())
except Exception as e:
    print(f'FAIL: {e}')

print('\n=== Ollama logs (last 40) ===')
!tail -40 {THESIS_ROOT}/logs/ollama_serve.log

print('\n=== GPU state ===')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


### Python processes (app / ollama)

In [ ]:
!curl -s http://localhost:11434/api/tags | python3 -c "import json,sys; d=json.load(sys.stdin); print('Models loaded:', [m['name'] for m in d.get('models', [])])"
!ps aux | grep -E "ollama|python3 app" | grep -v grep


### App log

In [ ]:
!tail -50 /tmp/app.log


### OPTIONAL: copy Ollama models from Drive to local SSD (faster first load)

In [ ]:
import os, shutil, subprocess, time, requests

# Stop ollama first
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(3)

# Copy models from Drive to local SSD
LOCAL_MODELS = '/content/ollama_models'
DRIVE_MODELS = f'{THESIS_ROOT}/ollama_models'

if not os.path.exists(LOCAL_MODELS):
    print(f'Copying {DRIVE_MODELS} → {LOCAL_MODELS}')
    print('(takes ~2-3 min for 11GB)')
    shutil.copytree(DRIVE_MODELS, LOCAL_MODELS)
print('✓ Done')
!du -sh {LOCAL_MODELS}

# Restart Ollama pointing to local models
env = os.environ.copy()
env['OLLAMA_MODELS'] = LOCAL_MODELS
log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'

with open(log_path, 'ab') as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'✓ Ollama ready after {i+1}s')
            print('Models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)


### OPTIONAL: FORCE rebuild of the ChromaDB index (deletes and rebuilds it in Drive!)

In [ ]:
import os, shutil, sys, time

DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# 1. Αφαίρεσε symlink
if os.path.islink(PROJECT_CHROMA):
    os.unlink(PROJECT_CHROMA)

# 2. Nuke completely with sleep για να αφήσει time το Drive να sync
if os.path.exists(DRIVE_CHROMA):
    shutil.rmtree(DRIVE_CHROMA)
    print('✓ Nuked Drive chroma')
time.sleep(3)

# 3. Verify διαγράφηκε
print('Still exists?', os.path.exists(DRIVE_CHROMA))

# 4. Recreate
os.makedirs(DRIVE_CHROMA)
os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)
print('✓ Fresh dirs')

# 5. Clear any Python-level chromadb caches
import importlib, sys
for mod in list(sys.modules.keys()):
    if 'chromadb' in mod or 'core.rag' in mod or mod == 'config':
        del sys.modules[mod]
print('✓ Cleared module cache')

# 6. Re-import fresh
sys.path.insert(0, PROJECT_DIR)
import config
import chromadb
from chromadb.utils import embedding_functions
import json

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config.RAG_EMBEDDING_MODEL
)
client = chromadb.PersistentClient(path=DRIVE_CHROMA)

# 7. Try delete first (idempotent)
try:
    client.delete_collection(config.CHROMA_COLLECTION_NAME)
    print(f'✓ Deleted existing collection')
except Exception as e:
    print(f'  (no existing collection to delete: {e})')

# 8. Create fresh with get_or_create as fallback
try:
    col = client.create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)
    print('✓ Created new collection')
except Exception as e:
    print(f'  create_collection failed: {e}')
    print('  Trying get_or_create...')
    col = client.get_or_create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)

# 9. Verify empty
print(f'Collection count before indexing: {col.count()}')

# 10. Load and index
CHUNKS_PATH = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
documents, ids, metadatas = [], [], []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        c = json.loads(line)
        documents.append(c['text'])
        ids.append(c['chunk_id'])
        metadatas.append({
            'source': c['source'],
            'volume': c['volume'],
            'chapter': c['chapter'],
            'section': c['section'],
            'page_start': c['page_start'],
            'page_end': c['page_end'],
            'license': c['license'],
            'n_tokens': c['n_tokens'],
        })

print(f'\n✓ Loaded {len(documents)} chunks, indexing...')
BATCH = 256
for i in range(0, len(documents), BATCH):
    col.add(
        documents=documents[i:i+BATCH],
        ids=ids[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )
    if (i // BATCH) % 4 == 0:
        print(f'[INDEX] {min(i+BATCH, len(documents))}/{len(documents)}')

print(f"\n[DONE] Collection '{config.CHROMA_COLLECTION_NAME}' now has {col.count()} chunks.")

# 11. CRITICAL verify - metadata exists
r = col.query(query_texts=["Newton second law"], n_results=1, include=["metadatas", "documents", "distances"])
print(f'\n=== VERIFY ===')
print(f'id: {r["ids"][0][0]}')
print(f'metadata: {r["metadatas"][0][0]}')


# 8. Stop the runtime (always at the end)

In [ ]:
# === STOP NOW: save Drive and delete the runtime ===
from google.colab import runtime, drive
try: drive.flush_and_unmount()
except Exception as e: print("Drive flush warning:", e)
runtime.unassign()